# Data Preprocess

Set `GENRE` to `"FPS"` or `"MIXED"` below, then run this notebook from the top.
The selected configuration determines the input games and output filename prefix.
Mixed processing preserves existing FPS split assignments for shared games such
as CS2, so existing FPS validation/test reviews do not enter mixed training.


In [1]:
import pandas as pd
from pathlib import Path
from sklearn.model_selection import train_test_split

from config import FPS_GAME_IDS, MIXED_GAME_IDS, EXTERNAL_TEST_GAME_IDS

DATA_DIR = Path("data/processed")
OUTPUT_DIR = Path("data/merged")
GENRE = "MIXED"  # Choose "FPS" or "MIXED".
SELECTED_GAME_IDS = {"FPS": FPS_GAME_IDS, "MIXED": MIXED_GAME_IDS}[GENRE]
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if set(SELECTED_GAME_IDS.values()) & set(EXTERNAL_TEST_GAME_IDS.values()):
    raise ValueError("External test games must remain outside training data.")


In [2]:
def load_reviews(data_dir, game_ids, genre):
    frames = []

    for game, appid in game_ids.items():
        path = data_dir / f"reviews_{game}_{appid}.csv"

        df = pd.read_csv(
            path,
            dtype={
                "recommendationid": "string",
                "review": "string",
                "voted_up": "boolean",
            },
        )

        df["game"] = game
        df["appid"] = appid
        df["genre"] = genre
        frames.append(df)

    return pd.concat(frames, ignore_index=True)

In [3]:
def preprocess_reviews(df):
    df = df.copy()

    # drop empty rows
    df = df.dropna(
        subset=["recommendationid", "review", "voted_up"]
    )

    df = df[df["review"].str.strip().ne("")]

    # drop duplicate reviews
    df = df.drop_duplicates(
        subset=["appid", "recommendationid"]
    )

    # recommended: 1; not recommended: 0
    df["voted_up"] = df["voted_up"].astype("int8")

    return df.reset_index(drop=True)

In [4]:
def inspect_reviews(df):
    # Inspect data without changing the DataFrame.
    print(f"Total reviews: {len(df)}")
    display(df.head())

    print("Missing values:")
    display(df.isna().sum().to_frame("missing_count"))

    text = (
        df["review"]
        .fillna("")
        .str.replace(r"\s+", " ", regex=True)
        .str.strip()
    )
    print("Empty or whitespace-only reviews:", int(text.eq("").sum()))
    print(
        "Duplicate review IDs:",
        int(df.duplicated(["appid", "recommendationid"]).sum()),
    )
    print(
        "Repeated text beyond the first occurrence (ignoring whitespace differences):",
        int(text[text.ne("")].duplicated().sum()),
    )

    print("Label distribution (0 = not recommended, 1 = recommended):")
    counts = df["voted_up"].value_counts().reindex([0, 1], fill_value=0)
    display(pd.DataFrame({
        "count": counts,
        "proportion": (counts / max(len(df), 1)).round(4),
    }))

    print("Label counts by game:")
    display(
        pd.crosstab(df["game"], df["voted_up"])
        .reindex(columns=[0, 1], fill_value=0)
        .rename(columns={0: "not_recommended", 1: "recommended"})
    )


In [5]:
reviews = load_reviews(DATA_DIR, SELECTED_GAME_IDS, GENRE)
reviews = preprocess_reviews(reviews)

inspect_reviews(reviews)

Total reviews: 20346
Missing values:
Empty or whitespace-only reviews: 0
Duplicate review IDs: 0
Repeated text beyond the first occurrence (ignoring whitespace differences): 644
Label distribution (0 = not recommended, 1 = recommended):
Label counts by game:


,recommendationid,review,voted_up,language,timestamp_created,timestamp_updated,playtime_at_review,game,appid,genre
0,239306808,fix your useless fucking anticheat which is ba...,0,english,2026-11-03 21:56:37,2026-11-03 22:05:54,16343,CS2,730,MIXED
1,238765240,"Cheats everywhere, maybe do something about it...",0,english,2026-10-28 18:50:59,2026-10-28 18:50:59,181830,CS2,730,MIXED
2,238777093,gay idiot stupid face game,0,english,2026-10-28 22:23:34,2026-10-28 22:23:34,4169,CS2,730,MIXED
3,238458264,i am not cheater,0,english,2026-10-24 14:02:41,2026-10-24 14:02:41,497,CS2,730,MIXED
4,239016472,I got banned wihtout cheating,0,english,2026-10-31 14:53:16,2026-10-31 14:53:16,578,CS2,730,MIXED


,missing_count
recommendationid,0
review,0
voted_up,0
language,0
timestamp_created,0
timestamp_updated,0
playtime_at_review,0
game,0
appid,0
genre,0


,count,proportion
voted_up,,
0,11584,0.5694
1,8762,0.4306


voted_up,not_recommended,recommended
game,,
AMERICAN_TRUCK_SIMULATOR,1678,1309
BLACK_MYTH_WUKONG,1743,1308
CS2,1431,933
DISCO_ELYSIUM,1820,1535
GTA5,1457,878
HADES,1729,1419
STREET_FIGHTER_6,1726,1380


In [6]:
# Split by game and label, retaining the existing FPS workflow.
reviews["stratify_label"] = reviews["game"].astype(str) + "_" + reviews["voted_up"].astype(str)

def split_reviews(frame):
    train, temporary = train_test_split(
        frame, test_size=0.2, stratify=frame["stratify_label"], random_state=42,
    )
    val, test = train_test_split(
        temporary, test_size=0.5, stratify=temporary["stratify_label"], random_state=42,
    )
    return train, val, test

if GENRE == "FPS":
    train_df, val_df, test_df = split_reviews(reviews)
else:
    split_frames = {split: [] for split in ("train", "val", "test")}
    for game, appid in SELECTED_GAME_IDS.items():
        frame = reviews[reviews["appid"].eq(appid)]
        if appid in FPS_GAME_IDS.values():
            # Reuse CS2's existing boundaries instead of moving old test rows to train.
            assigned = set()
            for split in split_frames:
                original = pd.read_csv(
                    OUTPUT_DIR / f"FPS_{split}.csv",
                    dtype={"recommendationid": "string"},
                )
                ids = set(original.loc[original["appid"].eq(appid), "recommendationid"])
                selected = frame[frame["recommendationid"].isin(ids)]
                if assigned & ids or len(selected) != len(ids):
                    raise ValueError(f"Check existing FPS split IDs for {game}.")
                assigned |= ids
                split_frames[split].append(selected)
            if assigned != set(frame["recommendationid"]):
                raise ValueError(f"Processed {game} reviews differ from existing FPS splits.")
        else:
            for split, selected in zip(split_frames, split_reviews(frame)):
                split_frames[split].append(selected)
    train_df, val_df, test_df = (
        pd.concat(frames, ignore_index=True) for frames in split_frames.values()
    )

# Check all assignments before saving.
seen = set()
for frame in (train_df, val_df, test_df):
    ids = set(zip(frame["appid"], frame["recommendationid"]))
    if len(ids) != len(frame) or seen & ids:
        raise ValueError("Review IDs overlap across splits.")
    seen |= ids
if len(seen) != len(reviews):
    raise ValueError("Some reviews were not assigned to a split.")

for split, frame in (("train", train_df), ("val", val_df), ("test", test_df)):
    frame.to_csv(OUTPUT_DIR / f"{GENRE}_{split}.csv", index=False)
    print(f"{GENRE} {split}: {len(frame)} reviews")


MIXED train: 16275 reviews
MIXED val: 2034 reviews
MIXED test: 2037 reviews


In [7]:
# save whole processed dataframe
output_path = OUTPUT_DIR / f"{GENRE}_reviews_merged.csv"

reviews.to_csv(output_path, index=False, encoding="utf-8")

print(f"Saved {len(reviews)} reviews to: {output_path}")

Saved 20346 reviews to: data\merged\MIXED_reviews_merged.csv
